# Scopus Title-Based Metadata Enrichment - Debugged Version

Notebook ini memakai **Elsevier Scopus Search API langsung via `requests`**, bukan `pybliometrics`.

Tujuan:

- mencari artikel di Scopus berdasarkan kolom `Title` saja;
- mengabaikan kolom `Tahun` karena tidak valid;
- mengisi kolom:
  - `Volume / Issue`
  - `LINK DOI/ARTIKEL`
  - `SCOPUS SITASI`
- menganggap tanda `-` sebagai kosong/NaN untuk `Volume / Issue` dan `LINK DOI/ARTIKEL`;
- menyimpan file hasil, log, dan checkpoint;
- memberi highlight kuning pada cell yang berhasil diisi.

Jalankan notebook ini dari atas ke bawah. Untuk uji coba pertama, biarkan `TEST_MODE = True`.

In [15]:
# Install dependency jika belum ada
# %pip install pandas openpyxl requests tqdm -q

In [16]:
import pandas as pd
import requests
import re
import time
import random
from pathlib import Path
from difflib import SequenceMatcher
from tqdm.auto import tqdm
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

## 1. Konfigurasi

Masukkan API key Elsevier kamu pada variabel `ELSEVIER_API_KEY`.

Catatan: jangan upload notebook berisi API key ke GitHub atau membagikannya ke orang lain.

In [ ]:
# =====================
# WAJIB DIISI
# =====================
ELSEVIER_API_KEY = ""

# =====================
# FILE INPUT / OUTPUT
# =====================
INPUT_FILE = "data 2026_combined.xlsx"
OUTPUT_FILE = "data_2026_scopus_enriched_title_only_debugged.xlsx"
LOG_FILE = "scopus_title_only_enrichment_log_debugged.xlsx"
CHECKPOINT_FILE = "checkpoint_data_2026_scopus_enriched.xlsx"
CHECKPOINT_LOG_FILE = "checkpoint_scopus_title_log.xlsx"

# =====================
# KOLOM
# =====================
TITLE_COL = "Title"
VOLUME_ISSUE_COL = "Volume / Issue"
ARTICLE_LINK_COL = "LINK DOI/ARTIKEL"
CITATION_COL = "SCOPUS SITASI"

# =====================
# MODE UJI COBA
# Untuk test awal, biarkan True dan 3 baris.
# Jika sudah aman, ubah TEST_MODE = False.
# =====================
TEST_MODE = False
TEST_N_ROWS = 3

# =====================
# PARAMETER API DAN MATCHING
# =====================
SCOPUS_SEARCH_URL = "https://api.elsevier.com/content/search/scopus"
COUNT_PER_QUERY = 5
REQUEST_TIMEOUT_SECONDS = 30
REQUEST_DELAY_SECONDS = 5      # jeda antarbaris; konservatif untuk menghindari rate limit
MAX_RETRIES = 3
SIMILARITY_THRESHOLD = 0.95    # jika title-only terlalu berisiko, naikkan ke 0.90
CHECKPOINT_EVERY = 25

assert ELSEVIER_API_KEY and ELSEVIER_API_KEY != "ISI_API_KEY_KAMU_DI_SINI", "Isi dulu ELSEVIER_API_KEY."

## 2. Load Excel dan standardisasi nilai kosong

In [18]:
input_path = Path(INPUT_FILE)
assert input_path.exists(), f"File tidak ditemukan: {input_path.resolve()}"

df = pd.read_excel(input_path)

required_cols = [TITLE_COL, VOLUME_ISSUE_COL, ARTICLE_LINK_COL, CITATION_COL]
missing_cols = [c for c in required_cols if c not in df.columns]
assert not missing_cols, f"Kolom wajib tidak ada: {missing_cols}"

# Anggap '-' sebagai kosong untuk kolom sesuai permintaan.
for col in [VOLUME_ISSUE_COL, ARTICLE_LINK_COL]:
    df[col] = df[col].replace({"-": pd.NA, "": pd.NA, "nan": pd.NA, "None": pd.NA})

print("Jumlah baris:", len(df))
print("Kosong per kolom target:")
print(df[[VOLUME_ISSUE_COL, ARTICLE_LINK_COL, CITATION_COL]].isna().sum())

Jumlah baris: 2279
Kosong per kolom target:
Volume / Issue       270
LINK DOI/ARTIKEL     228
SCOPUS SITASI       1524
dtype: int64


## 3. Fungsi utilitas: normalisasi title, similarity, format output

In [19]:
def normalize_title(text):
    # Normalisasi judul untuk perbandingan similarity.
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = text.replace("&amp;", " and ")
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def title_similarity(a, b):
    a_norm = normalize_title(a)
    b_norm = normalize_title(b)
    if not a_norm or not b_norm:
        return 0.0
    return SequenceMatcher(None, a_norm, b_norm).ratio()


def clean_title_for_query(title, max_len=280):
    # Membersihkan title agar aman dipakai di query Scopus.
    if pd.isna(title):
        return ""
    title = str(title)
    title = title.replace('"', " ")
    title = title.replace("“", " ").replace("”", " ")
    title = title.replace("‘", " ").replace("’", " ")
    title = re.sub(r"\s+", " ", title).strip()
    return title[:max_len]


def has_value(x):
    return x is not None and not pd.isna(x) and str(x).strip() != ""


def make_volume_issue(volume, issue):
    volume = None if pd.isna(volume) else str(volume).strip()
    issue = None if pd.isna(issue) else str(issue).strip()

    if volume and issue:
        return f"Vol. {volume}, Issue {issue}"
    elif volume:
        return f"Vol. {volume}"
    elif issue:
        return f"Issue {issue}"
    else:
        return pd.NA


def make_article_link(doi, eid=None):
    if has_value(doi):
        doi = str(doi).strip()
        if doi.lower().startswith("http"):
            return doi
        return f"https://doi.org/{doi}"
    if has_value(eid):
        return f"https://www.scopus.com/record/display.uri?eid={eid}"
    return pd.NA


def needs_enrichment(row):
    return (
        pd.isna(row.get(VOLUME_ISSUE_COL)) or
        pd.isna(row.get(ARTICLE_LINK_COL)) or
        pd.isna(row.get(CITATION_COL))
    )

## 4. Fungsi request Scopus API langsung via `requests`

Fungsi ini:

- memakai query `TITLE("judul lengkap")`;
- tidak memakai `Tahun`;
- mengambil maksimal 5 kandidat;
- memilih kandidat dengan similarity title tertinggi;
- menangani `429 Too Many Requests`, timeout, dan error sementara.

In [20]:
headers = {
    "X-ELS-APIKey": ELSEVIER_API_KEY,
    "Accept": "application/json"
}


def scopus_get_with_retry(params, max_retries=MAX_RETRIES):
    last_status = None
    last_text = ""
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(
                SCOPUS_SEARCH_URL,
                headers=headers,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS
            )
            last_status = r.status_code
            last_text = r.text[:500]

            if r.status_code == 200:
                return r, "OK"

            # Rate limit atau server sementara bermasalah: tunggu lebih lama lalu retry.
            if r.status_code in [429, 500, 502, 503, 504]:
                wait = (REQUEST_DELAY_SECONDS * attempt) + random.uniform(1, 3)
                print(f"HTTP {r.status_code}; retry {attempt}/{max_retries}; sleep {wait:.1f}s")
                time.sleep(wait)
                continue

            # Error lain tidak diretry.
            return r, f"HTTP_{r.status_code}: {last_text}"

        except requests.exceptions.Timeout:
            wait = (REQUEST_DELAY_SECONDS * attempt) + random.uniform(1, 3)
            print(f"TIMEOUT; retry {attempt}/{max_retries}; sleep {wait:.1f}s")
            time.sleep(wait)
        except Exception as e:
            return None, f"REQUEST_ERROR: {e}"

    return None, f"FAILED_AFTER_RETRIES_LAST_STATUS_{last_status}: {last_text}"


def search_scopus_title(title, count=COUNT_PER_QUERY, verbose=False):
    title_clean = clean_title_for_query(title)
    if not title_clean:
        return None, "EMPTY_TITLE"

    query = f'TITLE("{title_clean}")'
    params = {
        "query": query,
        "count": count,
        "start": 0
    }

    r, status = scopus_get_with_retry(params)
    if status != "OK":
        return None, status

    data = r.json()
    entries = data.get("search-results", {}).get("entry", [])
    if not entries:
        return None, "NOT_FOUND"

    candidates = []
    for item in entries:
        scopus_title = item.get("dc:title", "")
        sim = title_similarity(title, scopus_title)
        doi = item.get("prism:doi")
        eid = item.get("eid")
        volume = item.get("prism:volume")
        issue = item.get("prism:issueIdentifier")
        citedby = item.get("citedby-count")

        candidates.append({
            "title": scopus_title,
            "similarity": sim,
            "doi": doi,
            "eid": eid,
            "volume": volume,
            "issue": issue,
            "volume_issue": make_volume_issue(volume, issue),
            "article_link": make_article_link(doi, eid),
            "citedby": citedby,
            "query": query,
            "raw_link": item.get("link")
        })

    candidates = sorted(candidates, key=lambda x: x["similarity"], reverse=True)
    best = candidates[0]

    if verbose:
        print("Query:", query)
        print("Top candidate:", best["title"])
        print("Similarity:", best["similarity"])
        print("DOI:", best["doi"])
        print("Volume/Issue:", best["volume_issue"])
        print("Cited by:", best["citedby"])

    return best, "FOUND"

## 5. Tes koneksi API sangat ringan

Cell ini memastikan API key + VPN kampus bekerja. Harus menghasilkan status `200`.

In [21]:
test_params = {
    "query": 'TITLE("natural language processing") AND PUBYEAR = 2024',
    "count": 1,
    "start": 0
}

r, status = scopus_get_with_retry(test_params, max_retries=1)
print("Status helper:", status)
if r is not None:
    print("HTTP status:", r.status_code)
    print(r.text[:300])

Status helper: OK
HTTP status: 200
{"search-results":{"opensearch:totalResults":"1606","opensearch:startIndex":"0","opensearch:itemsPerPage":"1","opensearch:Query":{"@role": "request", "@searchTerms": "TITLE(\"natural language processing\") AND PUBYEAR = 2024", "@startPage": "0"},"link": [{"@_fa": "true", "@ref": "self", "@href": "ht


## 6. Tes satu title dari Excel

Ini memastikan fungsi title-only berjalan pada data asli.

In [22]:
sample_title = df[TITLE_COL].dropna().astype(str).str.strip()
sample_title = sample_title[sample_title != ""].iloc[0]

print("Judul yang dites:")
print(sample_title)

best, status = search_scopus_title(sample_title, verbose=True)
print("Status:", status)
best

Judul yang dites:
Ethnic identity and internal migration decision in Indonesia
Query: TITLE("Ethnic identity and internal migration decision in Indonesia")
Top candidate: Ethnic identity and internal migration decision in Indonesia
Similarity: 1.0
DOI: 10.1080/1369183X.2018.1561252
Volume/Issue: Vol. 46, Issue 13
Cited by: 24
Status: FOUND


{'title': 'Ethnic identity and internal migration decision in Indonesia',
 'similarity': 1.0,
 'doi': '10.1080/1369183X.2018.1561252',
 'eid': '2-s2.0-85060010639',
 'volume': '46',
 'issue': '13',
 'volume_issue': 'Vol. 46, Issue 13',
 'article_link': 'https://doi.org/10.1080/1369183X.2018.1561252',
 'citedby': '24',
 'query': 'TITLE("Ethnic identity and internal migration decision in Indonesia")',
 'raw_link': [{'@_fa': 'true',
   '@ref': 'self',
   '@href': 'https://api.elsevier.com/content/abstract/scopus_id/85060010639'},
  {'@_fa': 'true',
   '@ref': 'author-affiliation',
   '@href': 'https://api.elsevier.com/content/abstract/scopus_id/85060010639?field=author,affiliation'},
  {'@_fa': 'true',
   '@ref': 'scopus',
   '@href': 'https://www.scopus.com/inward/record.uri?partnerID=HzOxMe3b&scp=85060010639&origin=inward'},
  {'@_fa': 'true',
   '@ref': 'scopus-citedby',
   '@href': 'https://www.scopus.com/inward/citedby.uri?partnerID=HzOxMe3b&scp=85060010639&origin=inward'}]}

## 7. Tentukan baris yang akan diproses

Untuk uji coba awal, notebook hanya memproses 3 baris yang masih perlu enrichment. Kalau sudah berhasil, ubah `TEST_MODE = False` di konfigurasi paling atas, lalu restart dan jalankan ulang.

In [23]:
rows_to_process = df[df.apply(needs_enrichment, axis=1)].index.tolist()

print("Jumlah baris yang perlu enrichment sebelum test/full run:", len(rows_to_process))

if TEST_MODE:
    rows_to_process = rows_to_process[:TEST_N_ROWS]
    print(f"TEST_MODE aktif: hanya memproses {len(rows_to_process)} baris.")
else:
    print(f"FULL RUN: memproses {len(rows_to_process)} baris.")

rows_to_process[:10]

Jumlah baris yang perlu enrichment sebelum test/full run: 1785
FULL RUN: memproses 1785 baris.


[9, 11, 13, 14, 15, 16, 17, 18, 19, 24]

## 8. Jalankan enrichment

Status utama yang mungkin muncul:

- `UPDATED`: berhasil match dan ada update/setidaknya metadata tervalidasi;
- `LOW_SIMILARITY_NOT_UPDATED`: ditemukan kandidat tapi similarity di bawah threshold;
- `NOT_FOUND`: Scopus tidak mengembalikan kandidat;
- `HTTP_403`, `HTTP_429`, dll.: masalah akses/rate limit;
- `EMPTY_TITLE`: title kosong.

In [24]:
logs = []
changed_cells = set()

print("Mulai enrichment...")
print("Similarity threshold:", SIMILARITY_THRESHOLD)
print("Delay antarbaris:", REQUEST_DELAY_SECONDS, "detik")

for n, idx in enumerate(tqdm(rows_to_process, desc="Enriching"), start=1):
    original_title = df.at[idx, TITLE_COL]

    if pd.isna(original_title) or str(original_title).strip() == "":
        logs.append({
            "index": idx,
            "excel_row": idx + 2,
            "original_title": original_title,
            "status": "EMPTY_TITLE"
        })
        continue

    print(f"[{n}/{len(rows_to_process)}] index={idx}, title={str(original_title)[:90]}")

    best, status = search_scopus_title(original_title, verbose=False)

    if status != "FOUND" or not best:
        print("Status:", status)
        logs.append({
            "index": idx,
            "excel_row": idx + 2,
            "original_title": original_title,
            "status": status
        })
        time.sleep(REQUEST_DELAY_SECONDS)
        continue

    print("Matched:", best.get("title"))
    print("Similarity:", round(best.get("similarity", 0), 4))

    if best["similarity"] < SIMILARITY_THRESHOLD:
        print("Tidak diupdate: similarity rendah")
        logs.append({
            "index": idx,
            "excel_row": idx + 2,
            "original_title": original_title,
            "matched_title": best.get("title"),
            "similarity": best.get("similarity"),
            "doi": best.get("doi"),
            "eid": best.get("eid"),
            "status": "LOW_SIMILARITY_NOT_UPDATED"
        })
        time.sleep(REQUEST_DELAY_SECONDS)
        continue

    updated_columns = []

    if pd.isna(df.at[idx, VOLUME_ISSUE_COL]) and not pd.isna(best.get("volume_issue")):
        df.at[idx, VOLUME_ISSUE_COL] = best["volume_issue"]
        changed_cells.add((idx, VOLUME_ISSUE_COL))
        updated_columns.append(VOLUME_ISSUE_COL)

    if pd.isna(df.at[idx, ARTICLE_LINK_COL]) and not pd.isna(best.get("article_link")):
        df.at[idx, ARTICLE_LINK_COL] = best["article_link"]
        changed_cells.add((idx, ARTICLE_LINK_COL))
        updated_columns.append(ARTICLE_LINK_COL)

    if pd.isna(df.at[idx, CITATION_COL]) and best.get("citedby") is not None:
        df.at[idx, CITATION_COL] = best["citedby"]
        changed_cells.add((idx, CITATION_COL))
        updated_columns.append(CITATION_COL)

    print("Updated columns:", updated_columns if updated_columns else "Tidak ada cell kosong yang perlu diisi")

    logs.append({
        "index": idx,
        "excel_row": idx + 2,
        "original_title": original_title,
        "matched_title": best.get("title"),
        "similarity": best.get("similarity"),
        "doi": best.get("doi"),
        "eid": best.get("eid"),
        "volume": best.get("volume"),
        "issue": best.get("issue"),
        "volume_issue": best.get("volume_issue"),
        "article_link": best.get("article_link"),
        "citedby": best.get("citedby"),
        "updated_columns": ", ".join(updated_columns),
        "status": "UPDATED" if updated_columns else "MATCHED_NO_EMPTY_TARGET_CELL"
    })

    # Checkpoint berkala untuk full run.
    if (not TEST_MODE) and (n % CHECKPOINT_EVERY == 0):
        print(f"Checkpoint setelah {n} baris...")
        pd.DataFrame(logs).to_excel(CHECKPOINT_LOG_FILE, index=False)
        df.to_excel(CHECKPOINT_FILE, index=False)

    time.sleep(REQUEST_DELAY_SECONDS)

log_df = pd.DataFrame(logs)
print("Selesai loop.")
print("Jumlah cell berubah:", len(changed_cells))
print("Ringkasan status:")
print(log_df["status"].value_counts(dropna=False) if not log_df.empty else "Log kosong")

Mulai enrichment...
Similarity threshold: 0.95
Delay antarbaris: 5 detik


Enriching:   0%|          | 0/1785 [00:00<?, ?it/s]

[1/1785] index=9, title=The impact of auditors’ awareness of the profession’s reputation for independence on audit
Matched: 
Similarity: 0.0
Tidak diupdate: similarity rendah


Enriching:   0%|          | 1/1785 [00:06<2:59:46,  6.05s/it]

[2/1785] index=11, title=Anger punishes, compassion forgives: How discrete emotions mitigate double standards in co
Matched: Anger punishes, compassion forgives: How discrete emotions mitigate double standards in consumer ethical judgment
Similarity: 1.0
Updated columns: ['SCOPUS SITASI']


...

[1784/1785] index=2277, title=Work-Life Balance and Psychological Well-Being in Married Female Doctoral Students: A Phen
Matched: Work-Life Balance and Psychological Well-Being in Married Female Doctoral Students: A Phenomenological Study in Indonesian Higher Education
Similarity: 1.0
Updated columns: ['Volume / Issue']


Enriching: 100%|█████████▉| 1784/1785 [3:05:02<00:05,  5.68s/it]

[1785/1785] index=2278, title=The Role of Digital Technology Adoption in Strengthening Community Health Resilience: An I
Matched: The Role of Digital Technology Adoption in Strengthening Community Health Resilience: An Integrated TAM–TPB Model of Community Health Services in East Java
Similarity: 1.0
Updated columns: ['Volume / Issue']


Enriching: 100%|██████████| 1785/1785 [3:05:08<00:00,  6.22s/it]

Selesai loop.
Jumlah cell berubah: 1628
Ringkasan status:
status
UPDATED                                                                                                                                                    1467
LOW_SIMILARITY_NOT_UPDATED                                                                                                                                  280
MATCHED_NO_EMPTY_TARGET_CELL                                                                                                                                 26
FAILED_AFTER_RETRIES_LAST_STATUS_500: {"service-error":{"status":{"statusCode":"GENERAL_SYSTEM_ERROR","statusText":"Error transforming XML with XSL"}}}      11
REQUEST_ERROR: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))       1
Name: count, dtype: int64


## 9. Cek hasil sementara

In [25]:
if not log_df.empty:
    display(log_df.head(20))

print("Kosong per kolom target setelah run:")
print(df[[VOLUME_ISSUE_COL, ARTICLE_LINK_COL, CITATION_COL]].isna().sum())

,index,excel_row,original_title,matched_title,similarity,doi,eid,status,volume,issue,volume_issue,article_link,citedby,updated_columns
0,9,11,The impact of auditors’ awareness of the profe...,,0.0,None,None,LOW_SIMILARITY_NOT_UPDATED,NaN,NaN,NaN,NaN,NaN,NaN
1,11,13,"Anger punishes, compassion forgives: How discr...","Anger punishes, compassion forgives: How discr...",1.0,10.1016/j.jretconser.2019.101979,2-s2.0-85074063388,UPDATED,53,None,Vol. 53,https://doi.org/10.1016/j.jretconser.2019.101979,39,SCOPUS SITASI
2,13,15,Competitiveness and cost behaviour: evidence f...,Competitiveness and cost behaviour: evidence f...,1.0,10.1108/JAAR-08-2018-0120,2-s2.0-85075081806,UPDATED,21,1,"Vol. 21, Issue 1",https://doi.org/10.1108/JAAR-08-2018-0120,14,SCOPUS SITASI
3,14,16,Effects of Halal social media and customer eng...,Effects of Halal social media and customer eng...,1.0,10.1108/JIMA-06-2019-0119,2-s2.0-85075691692,UPDATED,11,6,"Vol. 11, Issue 6",https://doi.org/10.1108/JIMA-06-2019-0119,29,SCOPUS SITASI
4,15,17,Effects of Halal social media and customer eng...,Effects of Halal social media and customer eng...,1.0,10.1108/JIMA-06-2019-0119,2-s2.0-85075691692,UPDATED,11,6,"Vol. 11, Issue 6",https://doi.org/10.1108/JIMA-06-2019-0119,29,SCOPUS SITASI
5,16,18,Effects of Halal social media and customer eng...,Effects of Halal social media and customer eng...,1.0,10.1108/JIMA-06-2019-0119,2-s2.0-85075691692,UPDATED,11,6,"Vol. 11, Issue 6",https://doi.org/10.1108/JIMA-06-2019-0119,29,SCOPUS SITASI
6,17,19,Islamic microfinance institution: Survey data ...,Islamic microfinance institution: Survey data ...,1.0,10.1016/j.dib.2019.104911,2-s2.0-85076057590,UPDATED,28,None,Vol. 28,https://doi.org/10.1016/j.dib.2019.104911,4,SCOPUS SITASI
7,18,20,Explaining regional inflation programmes in In...,,0.0,None,None,LOW_SIMILARITY_NOT_UPDATED,NaN,NaN,NaN,NaN,NaN,NaN
8,19,21,Explaining regional inflation programmes in In...,,0.0,None,None,LOW_SIMILARITY_NOT_UPDATED,NaN,NaN,NaN,NaN,NaN,NaN
9,24,26,"Earnings management, business strategy, and ba...","Earnings management, business strategy, and ba...",1.0,10.1016/j.heliyon.2020.e03317,2-s2.0-85078759763,UPDATED,6,2,"Vol. 6, Issue 2",https://doi.org/10.1016/j.heliyon.2020.e03317,91,SCOPUS SITASI


Kosong per kolom target setelah run:
Volume / Issue      108
LINK DOI/ARTIKEL     17
SCOPUS SITASI       269
dtype: int64


## 10. Simpan hasil Excel + log + highlight cell yang diisi

Cell yang berhasil diisi akan diberi highlight kuning.

In [26]:
# Simpan file utama dan log
output_path = Path(OUTPUT_FILE)
log_path = Path(LOG_FILE)

df.to_excel(output_path, index=False)
log_df.to_excel(log_path, index=False)

# Highlight cell yang berubah
if changed_cells:
    wb = load_workbook(output_path)
    ws = wb.active
    fill = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")

    col_positions = {cell.value: cell.column for cell in ws[1]}

    for idx, col_name in changed_cells:
        excel_row = idx + 2  # header row + zero-based index
        excel_col = col_positions.get(col_name)
        if excel_col:
            ws.cell(row=excel_row, column=excel_col).fill = fill

    wb.save(output_path)

print("File hasil:", output_path.resolve())
print("File log:", log_path.resolve())
print("Jumlah cell di-highlight:", len(changed_cells))

File hasil: D:\Project\FEB\satya nyoba\data_2026_scopus_enriched_title_only_debugged.xlsx
File log: D:\Project\FEB\satya nyoba\scopus_title_only_enrichment_log_debugged.xlsx
Jumlah cell di-highlight: 1628


## 11. Langkah setelah test 3 baris berhasil

Jika test 3 baris berhasil:

1. ubah `TEST_MODE = False` pada cell konfigurasi;
2. sesuaikan `REQUEST_DELAY_SECONDS`, misalnya `5` detik untuk konservatif;
3. restart kernel;
4. jalankan ulang dari atas.

Estimasi kasar dengan delay 5 detik:

- 100 baris: sekitar 9-15 menit;
- 500 baris: sekitar 45-75 menit;
- 1.500 baris: sekitar 2-3 jam.

Jika sering mendapat `429`, naikkan `REQUEST_DELAY_SECONDS` ke 8-10 detik.